In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-13 12:59:03--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-09-13 12:59:03 (29.4 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [ ]:
#Let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
#Here are all the unique characters that occurs in this text
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
#Create a mapping from characters to integers
#This is the tokenizer we are gonna use
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
encode=lambda s: [stoi[c] for c in s] #encoder:takes a string,output a list of integers
decode=lambda l: ''.join([itos[i] for i in l]) #decoder:takes a list of integers,output a string

In [ ]:
print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
#Lets now encode the entire dtaset and store in into a torch.Tensor
import torch
data=torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:1000]) #the 1000 characters we looked at earlier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [ ]:
#Lets now split the data into the train and validation sets
n=int(0.9*len(data)) #first 90% will be train and rest val
train_data=data[:n]
val_data=data[n:]

In [ ]:
#We are never gonna feed the whole dataset into the model
#We always give chunks of data

In [ ]:
block_size=8
train_data[:block_size+1]
#why block_size+1??

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
#both x and y have length (8)
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
  context=x[:t+1]
  target=y[t]
  print(f"when input is {context} the target: {target}")
#this is why we used block_size+1 inorder to get our 8 examples
#To supervise the prediction when given all 8 tokens as context,
#you need that 9th token to serve as the final target

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [ ]:
#We are sending these things into batches of data so as to do parallel processing

The Function Signature: torch.randint(high, size)First argument (high): len(data) - block_sizeThis sets the upper bound (exclusive). The generated random integers will be in the range:$$[0, \text{len}(data) - \text{block\_size} - 1]$$Second argument (size): (batch_size,)This is the shape of the tensor to create. Passing (4,) tells PyTorch to return 1D tensor containing 4 random integers.

In [ ]:
torch.manual_seed(1337)
batch_size=4 #no of independent sequences we will process in parallel?
block_size=8 #what is the maximum context length of predictions?

def get_batch(split):
  #generate a small batch of data of inputs x and targets y
  data=train_data if split=='train' else val_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  #The 4 random integers acts as starting points for 4 diffent context sequences
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y
xb,yb=get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size):
  for t in range(block_size):
    context=xb[b,:t+1]
    target=yb[b,t]
    print(f"when input is {context.tolist()} the target is: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] the target is: 43
when input is [24, 43] the target is: 58
when input is [24, 43, 58] the target is: 5
when input is [24, 43, 58, 5] the target is: 57
when input is [24, 43, 58, 5, 57] the target is: 1
when input is [24, 43, 58, 5, 57, 1] the target is: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is: 39
when input is [44] the target is: 53
when input is [44, 53] the target is: 56
when input is [44, 53, 56] the target is: 1
when input is [44, 53, 56, 1] the target is: 58
when input is [44, 53, 56, 1, 58]

When passing xb with shape (B, T) = (4, 8) into self.token_embedding_table:Every integer in xb acts as an index into the table.For each integer $k \in [0, \text{vocab\_size}-1]$, PyTorch retrieves row $k$ from the weight matrix (a vector of length vocab_size).The resulting tensor shape is:$$\text{Shape: } (B, T, C) = (4, 8, \text{vocab\_size})$$Here, $C = \text{vocab\_size}$ represents the class scores for the next predicted token
idx	(4, 8)	4 independent sequences of 8 characters each
logits	(4, 8, 65)	For each position in each sequence, a 65-dimensional score predicting what character comes next

self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
Underneath the hood, nn.Embedding(vocab_size, vocab_size) is literally just a 2D float tensor of weights of shape (vocab_size, vocab_size).Suppose vocab_size = 65. This is a $65 \times 65$ grid of numbers initialized randomly.Row $i$ represents token index $i$.The 65 numbers inside row $i$ represent the raw unnormalized scores (logits) for what token should come immediately after token $i$.Column 0: score for predicting 'a'Column 1: score for predicting 'b'...Column 64: score for predicting '\n'

this is where the magic happens. How does an embedding table handle a 2D tensor idx?PyTorch loops over all $B \times T = 32$ numbers in idx completely independently.For each number at position (b, t):It takes that integer, say idx[0, 3] = 42.It goes directly to row 42 of token_embedding_table.weight.It pulls out all 65 numbers stored in row 42.It places those 65 numbers into logits[0, 3, :].Because it replaces every single integer with a vector of length $C$ ($C = \text{vocab\_size} = 65$), the shape becomes:$$(B, T) \longrightarrow (B, T, C) = (4, 8, 65)$$

In [ ]:
#Lets start with the simplest neural network:Bigram model
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    #each token directly reads off teh logits for the next token from a lookup table
    self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)

  def forward(self,idx,targets=None):
    #idx and targets both (B,T) tensor of integers
    logits=self.token_embedding_table(idx) #B,T,C
    if targets is None:
      loss=None
    else:
      #Pytorch wants inputs as B,C,T
      B,T,C=logits.shape
      logits=logits.view(B*T,C)
      targets=targets.view(B*T)
      loss=F.cross_entropy(logits,targets)
    return logits,loss

  def generate(self,idx,max_new_tokens):
    #idx is (B,T) array of indices in the current context
    for _ in range(max_new_tokens):
      #get the prediction:
      logits,loss=self(idx)
      #focus only on the last time step
      logits=logits[:,-1,:] #becomes B,C
      #apply softmax to get probabilities
      probs=F.softmax(logits,dim=-1) #(B,C)
      #sample from the distribution
      idx_next=torch.multinomial(probs,num_samples=1)#(B,1)
      #append sample index to the running seuquence
      idx=torch.cat((idx,idx_next),dim=1) #{B,T+1}
    return idx

n=BigramLanguageModel(vocab_size)
logits,loss=n(xb,yb)
print(logits.shape)
print(loss)
idx=torch.zeros((3,1),dtype=torch.long)
out=n.generate(idx,max_new_tokens=100)
print(out.shape)
for b in range(out.shape[0]):
  print(f"--- SAMPLE {b+1} ---")
  print(decode(out[b].tolist()))
  print()

torch.Size([256, 65])
tensor(4.6523, grad_fn=<NllLossBackward0>)
torch.Size([3, 101])
--- SAMPLE 1 ---

SPTEoi nk3SP :w3HMbjbLvOUZYWWNDEQJ:C?Ne:&u-jdRToCphiXiYF$'T:q.?kznV&yh&dSnepRwUBjRJrQC aZR?YZ3viVG,!

--- SAMPLE 2 ---

Sq,LTEXGjFqKqlgwOTE!uw?OGwfxY-uOLtB&l,ObcYWz dZ;eglzkQ:iOL&D cqDn3:3:hPTbhFMi
Sx?Od'nWrI&P,lgc-nLTAU

--- SAMPLE 3 ---

STEM:&Ac3QTX;PFkgCH3feD fYCVetXNf?pNpR3xwWxAIo?puoX-U;q,BkSgX: fqRSBkRvvvJJAcegjiGUEJmIM;qgRjub-!BfM



In [ ]:
#Lets Train this model
#Create a PyTorch Optimizer
optimizer=torch.optim.AdamW(n.parameters(),lr=1e-2)

In [ ]:
batch_size=32
for steps in range(1000):
  #sample a batch of data
  xb,yb=get_batch('train')
  #evaluate the loss
  logits,loss=n(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())

2.4539577960968018


In [ ]:
out=n.generate(idx,max_new_tokens=100)
print(out.shape)
for b in range(out.shape[0]):
  print(f"--- SAMPLE {b+1} ---")
  print(decode(out[b].tolist()))
  print()

torch.Size([3, 101])
--- SAMPLE 1 ---

AUCIUC-VOLqCE:
BR!
Ahmy th I noug ucerald themymog br oraly hirt mont I n me l?
ToraHAn stivef Buter

--- SAMPLE 2 ---


ERWhodsotrlocholefrirs t
They$f sby-G y fakeracasthinad
W:
KIORUGRieotimy, es, st he wesalyor fous,

--- SAMPLE 3 ---

&j beleacloner?

Bo 'sty doncang, that beanr med:TUKI I withe is triq$s, bonFo If?
Soten w t it my o



The Mathematical Trick in Self Attention

In [ ]:
#consider the following example
torch.manual_seed(1337)
B,T,C=(4,8,2)
x=torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
#We want the tokens to interact with each other
#While making sure that the previous token do not look for future tokens
#Each token just interacts with its pervious tokens only
#The simplest way for them to interact is taking average of all previous tokens
#We want x[b,t]=mean(i<=t) x[b,i]

##This is just the masked self attention we learned earlier but its
#using their mean for the current rather the weighted sum of Key,value

xbow=torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev=x[b,:t+1] #{t,c}
    xbow[b,t]=torch.mean(xprev,0)


In [ ]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [ ]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

et sequence length $T = 3$, and let each token be represented by a feature vector of size $C$:$$x = \begin{bmatrix} \mathbf{x}_0 \\ \mathbf{x}_1 \\ \mathbf{x}_2 \end{bmatrix} \quad (\text{shape: } 3 \times C)$$To compute the running sum of all past tokens at every position $t$, you multiply on the left by a lower-triangular matrix of ones:$$\begin{bmatrix} 1 & 0 & 0 \\ 1 & 1 & 0 \\ 1 & 1 & 1 \end{bmatrix} \begin{bmatrix} \mathbf{x}_0 \\ \mathbf{x}_1 \\ \mathbf{x}_2 \end{bmatrix} = \begin{bmatrix} \mathbf{x}_0 \\ \mathbf{x}_0 + \mathbf{x}_1 \\ \mathbf{x}_0 + \mathbf{x}_1 + \mathbf{x}_2 \end{bmatrix}$$Row 0 of the weight matrix computes $1 \cdot \mathbf{x}_0 + 0 \cdot \mathbf{x}_1 + 0 \cdot \mathbf{x}_2 = \mathbf{x}_0$.Row 1 computes $1 \cdot \mathbf{x}_0 + 1 \cdot \mathbf{x}_1 + 0 \cdot \mathbf{x}_2 = \mathbf{x}_0 + \mathbf{x}_1$.Row 2 computes $1 \cdot \mathbf{x}_0 + 1 \cdot \mathbf{x}_1 + 1 \cdot \mathbf{x}_2 = \mathbf{x}_0 + \mathbf{x}_1 + \mathbf{x}_2$.To turn those cumulative sums into averages, each row is divided by its number of active elements:$$W = \begin{bmatrix} 1.0 & 0 & 0 \\ 0.5 & 0.5 & 0 \\ 0.333 & 0.333 & 0.333 \end{bmatrix}$$Now, $W @ x$ computes the exact progressive average for every token position.

In [ ]:
torch.tril((torch.ones(3,3)))
#tril:triangular
#since we need mean we just divide by no. of active components
a=torch.tril((torch.ones(3,3)))
a=a/torch.sum(a,1,keepdims=True)
a

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

In [ ]:
wei=torch.tril((torch.ones(T,T))) #T->sequence length B->batches ,C=>embedding dimension
wei=wei/torch.sum(wei,1,keepdims=True)
#x has dimension B,T,C
xbow2=wei@x
xbow2 #(T,T) @ (B,T,C) --> (B,T,C)

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [ ]:
xbow.shape

torch.Size([4, 8, 2])

In [ ]:
xbow2.shape

torch.Size([4, 8, 2])

In [ ]:
#version 3 Softmax
tril=torch.tril(torch.ones(T,T))
wei=torch.zeros((T,T))
wei=wei.masked_fill(tril==0,float('-inf'))
wei=F.softmax(wei,dim=-1) #softmax along rows
xbow3=wei@x
torch.allclose(xbow2,xbow3)

#We use this version because these weights would start as 0 but gets updated by backprop
#Which is the say of each previous token for the next token but using k,q,v

True

In [ ]:
#version 4:Self Attention
torch.manual_seed(1337)
B,T,C=4,8,32 #batch,time,channels
x=torch.randn(B,T,C)

#Let's see a single Head Perform self attention
nhead=1
n_embd=x.shape[2]
dk=n_embd//nhead
key=nn.Linear(C,dk,bias=False)
query=nn.Linear(C,dk,bias=False)
value=nn.Linear(C,dk,bias=False)
k=key(x) #B,T,dk
q=query(x)
v=value(x)

wei=q@k.transpose(-2,-1) #only transpose the last two dimension B,T,dk @ B,dk,T ->B,T,T
tril=torch.tril(torch.ones(T,T))
wei=wei.masked_fill(tril==0,float('-inf'))
wei=F.softmax(wei/dk**0.5,dim=-1)
out=wei@v
out.shape

torch.Size([4, 8, 32])

In [ ]:
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [ ]:
wei[0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [8.3841e-01, 1.6159e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [8.8476e-01, 1.0445e-01, 1.0783e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [9.5338e-02, 6.5157e-01, 1.8913e-01, 6.3963e-02, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [8.6153e-01, 2.7327e-02, 3.9553e-03, 1.6660e-02, 9.0524e-02, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [4.6015e-05, 3.2064e-05, 1.1772e-02, 4.2589e-01, 2.5680e-06, 5.6226e-01,
         0.0000e+00, 0.0000e+00],
        [5.4061e-02, 6.4071e-02, 2.5209e-01, 8.7270e-02, 2.3189e-01, 2.3952e-01,
         7.1099e-02, 0.0000e+00],
        [5.4092e-02, 1.0762e-01, 1.6648e-02, 1.6889e-01, 6.2352e-01, 6.3775e-03,
         9.1127e-03, 1.3737e-02]], grad_fn=<SelectBackward0>)

In [ ]:
#Layer Norm 1D
class LayerNorm1d:

  def __init__(self,dim,eps=1e-5,momentum=0.1):
    self.eps=eps
    self.momentum=momentum
    self.training=True
    #parameters {trained with backprop}
    self.gamma=torch.ones(dim)
    self.beta=torch.zeros(dim);
    #buffers {trained with a running momentum update}
    self.running_mean=torch.zeros(dim)
    self.running_var=torch.ones(dim)

  def __call__(self,x):
    #calculate the forward pass
    xmean=x.mean(1,keepdim=True) #batchmean
    xvar=x.var(1,keepdim=True,unbiased=True) #batch varience
    xhat=(x-xmean)/torch.sqrt(xvar+self.eps) #normalize to unit varience
    self.out=self.gamma*xhat+self.beta
    # #update the buffers : because we always get data point there is no need to maintain running mean
    # its unique for each row we get calculated in hand
    # if self.training:
    #   with torch.no_grad():
    #     self.running_mean=(1-self.momentum)*self.running_mean+self.momentum*xmean
    #     self.running_var=(1-self.momentum)*self.running_var+self.momentum*xvar
    return self.out

  def parameters(self):
    return [self.gamma,self.beta]
torch.manual_seed(1337)
module=LayerNorm1d(100)
x=torch.randn(32,100)#batch size of 32 100 dimensional vectors
x=module(x)
x.shape

torch.Size([32, 100])

In [ ]:
x[:,0].mean(),x[:,0].std() #mean,std of one feature across all batches
#so it basically normalizes along columns

(tensor(0.1469), tensor(0.8803))

In [ ]:
x[0,:].mean(),x[0,:].std()
#it normalizes along rows

(tensor(-9.5367e-09), tensor(1.0000))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#hyper parameters
batch_size=64 #how many independent subsequences to process in parallel?
block_size=256 #what is the maximum context length for predictions?
max_iters=5000
eval_interval=500
learning_rate=3e-4
device='cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
  print(f"GPU: {torch.cuda.get_device_name(0)}")
eval_iters=200
n_embd=384
n_head=6
n_layer=6
dropout=0.2


torch.manual_seed(1337)

# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars=sorted(list(set(text)))
vocab_size=len(chars)
#Create a mapping from characters to integers
#This is the tokenizer we are gonna use
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
encode=lambda s: [stoi[c] for c in s] #encoder:takes a string,output a list of integers
decode=lambda l: ''.join([itos[i] for i in l]) #decoder:takes a list of integers,output a string

data=torch.tensor(encode(text),dtype=torch.long)
#Lets now split the data into the train and validation sets
n=int(0.9*len(data)) #first 90% will be train and rest val
train_data=data[:n]
val_data=data[n:]

#data loading
def get_batch(split):
  #generate a small batch of data of inputs x and targets y
  data=train_data if split=='train' else val_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  #The 4 random integers acts as starting points for 4 diffent context sequences
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+block_size+1] for i in ix])
  x,y=x.to(device),y.to(device)
  return x,y


@torch.no_grad()
#For finding the loss of the model we change the model to eval and find the losses over the data
#and again change the mode of model to train
def estimate_loss():
    out={}
    model.eval()
    for split in ['train','val']:
        losses=torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y=get_batch(split)
            logits,loss=model(X,Y)
            losses[k]=loss.item()
        out[split]=losses.mean()
    model.train()
    return out

class Head(nn.Module):
  def __init__(self,dk):
   super().__init__()
   self.key=nn.Linear(n_embd,dk,bias=False)
   self.query=nn.Linear(n_embd,dk,bias=False)
   self.value=nn.Linear(n_embd,dk,bias=False)
   self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
   self.dk=dk
   #self.register_buffer tells PyTorch: "This tensor is a persistent part of the module's state, but it is NOT a trainable parameter."
   #can't use a simple variable because it is ignored when we save the model
   self.dropout=nn.Dropout(dropout)


  def forward(self,x):
   B,T,C=x.shape
   k=self.key(x)
   q=self.query(x)
   #compute attention scores
   wei=q@k.transpose(-2,-1)*self.dk**-0.5
   wei=wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))
   wei=F.softmax(wei,dim=-1)
   wei=self.dropout(wei)
   v=self.value(x)
   out=wei@v
   return out

class MultiHeadAttention(nn.Module):

   def __init__(self,num_heads,dk):
      super().__init__()
      self.heads=nn.ModuleList([Head(dk) for _ in range(num_heads)])
      self.proj=nn.Linear(n_embd,n_embd) #this is Wo we studied in the notes
      self.dropout=nn.Dropout(dropout) #added for optimization

   def forward(self,x):
      out=torch.cat([h(x) for h in self.heads],dim=-1)
      out=self.dropout(self.proj(out))
      return out

class FeedForward(nn.Module):
  def __init__(self,n_embd):
      super().__init__()
      self.net=nn.Sequential(
         nn.Linear(n_embd,4*n_embd),
         nn.ReLU(),
         nn.Linear(4*n_embd,n_embd),
         nn.Dropout(dropout)
      )

  def forward(self,x):
     return self.net(x)

class Block(nn.Module):

  def __init__(self,n_embd,n_head):
      super().__init__()
      dk=n_embd//n_head
      self.sa=MultiHeadAttention(n_head,dk)
      self.ffwd=FeedForward(n_embd)
      self.ln1=nn.LayerNorm(n_embd)
      self.ln2=nn.LayerNorm(n_embd)
  def forward(self,x):
     x=x+self.sa(self.ln1(x)) #This is the only slight change
     #Nowdays first the layer norm is applied and then residual
     #component is added
     x=x+self.ffwd(self.ln2(x))
     return x

#Super Simple Biagram Model
class BigramLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()
    #each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table=nn.Embedding(vocab_size,n_embd)
    self.position_embedding_table=nn.Embedding(block_size,n_embd)
    self.blocks=nn.Sequential(*[Block(n_embd,n_head=n_head) for _ in range(n_layer)])
    self.ln_f=nn.LayerNorm(n_embd) #final Layer Norm
    self.lm_head=nn.Linear(n_embd,vocab_size)

  def forward(self,idx,targets=None):
    #idx and targets both (B,T) tensor of integers
    B,T=idx.shape
    token_embd=self.token_embedding_table(idx) #B,T,C
    pos_embd=self.position_embedding_table(torch.arange(T,device=device)) #T,C
    x=token_embd+pos_embd
    x=self.blocks(x)
    x=self.ln_f(x)
    logits=self.lm_head(x) #B,T,vocab_size
    if targets is None:
      loss=None
    else:
      #Pytorch wants inputs as B,C,T
      B,T,C=logits.shape
      logits=logits.view(B*T,C)
      targets=targets.view(B*T)
      loss=F.cross_entropy(logits,targets)
    return logits,loss

  def generate(self,idx,max_new_tokens):
    #idx is (B,T) array of indices in the current context
    for _ in range(max_new_tokens):
      #since we are usign position encoding table of size block_size
      #if we get a sequence with length>block size then we can't process that
      #crop idx to the last block_size tokens remove the past ones
      idx_cond=idx[:,-block_size:]
      #get the prediction:
      logits,loss=self(idx_cond)
      #focus only on the last time step
      logits=logits[:,-1,:] #becomes B,C
      #apply softmax to get probabilities
      probs=F.softmax(logits,dim=-1) #(B,C)
      #sample from the distribution
      idx_next=torch.multinomial(probs,num_samples=1)#(B,1)
      #append sample index to the running seuquence
      idx=torch.cat((idx,idx_next),dim=1) #{B,T+1}
    return idx

model=BigramLanguageModel()
m=model.to(device)

#create a pytorch optimizer
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)

for iter in range(max_iters):

    #every once in a while evaluate the loss on train and val sets so we calculate average loss over batches
    #because one batch the loss might get lucky and hence we do it give average loss after every few iterations
    #to get a rough idea of how loss is changing with training
    if iter%eval_interval==0:
        losses=estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f},val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb=get_batch('train')
    #evaluate the loss
    logits,loss=model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

#Generate from the model:
context=torch.zeros((1,1),dtype=torch.long,device=device)
print(decode(m.generate(context,max_new_tokens=500)[0].tolist()))



Device: cuda
GPU: Tesla T4
--2026-09-13 13:27:52--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-09-13 13:27:53 (56.6 MB/s) - ‘input.txt’ saved [1115394/1115394]

step 0: train loss 4.2846,val loss 4.2820
step 500: train loss 1.8867,val loss 2.0024
step 1000: train loss 1.5360,val loss 1.7221
step 1500: train loss 1.3948,val loss 1.6027
step 2000: train loss 1.3078,val loss 1.5497
step 2500: train loss 1.2514,val loss 1.5151
step 3000: train loss 1.2004,val loss 1.4903
step 3500: train loss 1.1584,val loss 1.4794
step 4000: train l